# Testing Context Extraction with LLM

In [4]:
# imports
from openai import OpenAI
from urllib import request
import json


## Test Examples

In [5]:
fileCode1 = """
# Globale Variablen
num_a = 10
num_b = 20
c = 30

# Funktionen
def func1():
    return num_a + helper1()

def helper1():
    return num_b + helper2()

def helper2():
    return 42

def dummy1():
    test = 5;
    return test
"""

functionToAnalyze1 = "func1"

fileCode2 = """
# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

def dummy2():
    return unused
"""

functionToAnalyze2 = "func2"

fileCode3 = """
# Globale Variablen
threshold = 100
step = 3
start = 1

# Funktionen
def func3():
    value = start
    while value < threshold:
        value = helper3(value)
    return value

def helper3(x):
    return x + step

def dummy3():
    return threshold
"""

functionToAnalyze3 = "func3"

fileCode4 = """
# Globale Variablen
base = 0
decrement = 1

# Funktionen
def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base

def helper_unused():
    return decrement
"""

functionToAnalyze4 = "func4"

fileCode5 = """
# Globale Variablen
MAX = 100
MIN = 1
STEP = 5
factor = 2
unused = 999

# Funktionen
def func5(n):
    total = 0
    for i in range(MIN, n, STEP):
        if i % 2 == 0:
            total += helper_even(i)
        else:
            total += helper_odd(i)
    while total < MAX:
        total = helper_increment(total)
    return total

def helper_even(x):
    return x * factor + helper_increment(x)

def helper_odd(x):
    if x % 3 == 0:
        return x // 2
    else:
        return x + factor

def helper_increment(value):
    if value >= MAX:
        return value
    return value + STEP

def dummy5():
    temp = 42
    return unused
"""

functionToAnalyze5 = "func5"

fileCode6 = """
# Globale Variablen
LIMIT = 50
OFFSET = 3
factor = 2
unused_global = 999

def ufunc7():
    return "I am not used"

# Funktionen
def func6(x):
    result = 0
    for i in range(x):
        if i % 2 == 0:
            result += even(i)
        elif i % 3 == 0:
            result += helper_multiple_three(i)
        else:
            result += helper_odd(i)
    return result

def even(n):
    return n * factor

def helper_odd(n):
    return n + OFFSET

def helper_multiple_three(n):
    if n > LIMIT:
        return LIMIT
    return n * 2

def dummy6():
    temp = 123
    return unused_global
"""

functionToAnalyze6 = "func6"

fileCode7 = """
# Globale Variablen
BASE = 10
STEP = 2
MAX = 100
factor = 3

# Funktionen
def dummy7():
    return 42  # Nicht genutzt

def func7(n):
    total = 0
    while n > 0:
        total += process_step(n)
        n -= STEP
    return finalize(total)

def process_step(value):
    if value % 5 == 0:
        return helper5(value)
    elif value % 3 == 0:
        return helper3(value)
    else:
        return value + factor

def helper5(v):
    return v // 5

def helper3(v):
    return v // 3

def finalize(result):
    return result % MAX

def unused_helper():
    return 0
"""

functionToAnalyze7 = "func7"



In [18]:
client = OpenAI()

def extract_context_gpt52(full_code: str, target_fn: str):
    response = client.responses.create(
        model="gpt-5.2",
        temperature=0.0,
        input=[
            {
                "role": "system",
                "content": (
                    "You are a static program analysis assistant. "
                    "You reason over Python source code precisely and deterministically."
                )
            },
            {
                "role": "user",
                "content": f"""
Full source code:
{full_code}

Target function:
{target_fn}

Task:
1. Identify ALL functions that are directly or indirectly called by the target function.
2. Identify ALL global variables used by those functions.
3. Extract ONLY:
   - the definitions of those global variables
   - the definitions of those functions
4. Sort the extracted functions topologically:
   - functions without dependencies first
   - callers after callees

Rules:
- Output valid Python Code.
- Do NOT include unused functions or globals.
- Do NOT include comments, explanations, or markdown.
- Preserve exact Python syntax and indentation.
"""
            }
        ]
    )

    # Raw Text Output holen
    raw_text = response.output_text

    return raw_text


In [26]:
fullFileCode = fileCode7
finalFunctionToAnalyse = functionToAnalyze7

context = extract_context_gpt52(fullFileCode, finalFunctionToAnalyse)

print(context)


STEP = 2
MAX = 100
factor = 3

def helper5(v):
    return v // 5

def helper3(v):
    return v // 3

def finalize(result):
    return result % MAX

def process_step(value):
    if value % 5 == 0:
        return helper5(value)
    elif value % 3 == 0:
        return helper3(value)
    else:
        return value + factor

def func7(n):
    total = 0
    while n > 0:
        total += process_step(n)
        n -= STEP
    return finalize(total)
